# PortalPulse Module 4: Tactical Action Intelligence Policy Network
## Hardware-Accelerated Deep PyTorch MLP for Prescriptive SRE Decision-Making

### Strategic Overview
This notebook trains and benchmarks the **PortalPulse Action Intelligence Policy Network**, which sits at the heart of the closed-loop autonomous control tower. Given a unified 13-dimensional state vector (telemetry, M2 multi-horizon load forecast, and M3/M4 anomaly & RCA diagnostics), the model prescribes the optimal tactical SRE action across **11 discrete mitigation strategies** codified from `secure_action.md`.

### Action Catalog (11 Discrete SRE Actions):
0. `NO_OP` - System nominal; continuous passive monitoring
1. `SCALE_OUT_WORKERS` - Add worker pods horizontally via Kubernetes HPA
2. `SCALE_IN_WORKERS` - Consolidate idle pods during low traffic to reduce cost
3. `ACTIVATE_RATE_LIMIT` - Edge token-bucket ingress rate limiting
4. `INCREASE_DB_POOL` - Expand PostgreSQL connection pool ceiling
5. `PURGE_REDIS_CACHE` - Invalidate and proactively refresh Redis cache
6. `DRAIN_REQUEST_QUEUE` - Shed non-critical queued tasks
7. `ENGAGE_CIRCUIT_BREAKER` - Trip failing upstream dependency
8. `RESTART_DEGRADED_PODS` - Rolling restart of memory-leaked pods
9. `DIVERT_READS_TO_REPLICA` - Offload read queries to read-replicas
10. `ENABLE_WAITING_ROOM` - Dynamic FIFO virtual waiting room


In [ ]:
import os
import sys
import time
import json
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from backend.m4_action_intelligence.dataset_generator import generate_action_dataset
from backend.m4_action_intelligence.policy_net import ActionPolicyMLP, get_device
from backend.m4_action_intelligence.schema import ACTION_INDEX_MAP, STATE_FEATURE_NAMES, SREAction

print(f"PyTorch Version: {torch.__version__}")
device = get_device()
print(f"Active Compute Hardware: {device}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Dedicated VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.1f} MB")


In [ ]:
# Step 1: Synthesize 25,000 State-Action Telemetry Records
print("Generating 25,000 synthetic state-action training samples...")
X, y, df = generate_action_dataset(n_samples=25000, random_state=42)
print(f"Dataset generated successfully! Feature matrix: {X.shape}, Labels: {y.shape}")
df.head()

In [ ]:
# Step 2: Stratified 80/10/10 Train/Validation/Test Splits & Standardization
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")


In [ ]:
# Step 3: Instantiate PyTorch Deep ActionPolicyMLP Network
model = ActionPolicyMLP(
    input_dim=13,
    hidden_dims=(128, 256, 128, 64),
    output_dim=11,
    dropout_rates=(0.20, 0.25, 0.15)
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")


In [ ]:
# Step 4: PyTorch Training Loop with AdamW and Cosine Annealing
batch_size = 64
epochs = 20

train_dataset = TensorDataset(torch.from_numpy(X_train_scaled), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val_scaled), torch.from_numpy(y_val))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print("Starting training on device:", device)
for epoch in range(1, epochs + 1):
    model.train()
    t_loss, t_corr, t_total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * len(by)
        t_corr += (torch.argmax(logits, -1) == by).sum().item()
        t_total += len(by)
    scheduler.step()

    model.eval()
    v_loss, v_corr, v_total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            v_loss += loss.item() * len(by)
            v_corr += (torch.argmax(logits, -1) == by).sum().item()
            v_total += len(by)

    history["train_loss"].append(t_loss / t_total)
    history["val_loss"].append(v_loss / v_total)
    history["train_acc"].append(t_corr / t_total)
    history["val_acc"].append(v_corr / v_total)

    if epoch % 5 == 0 or epoch == epochs:
        print(f"Epoch {epoch:2d}/{epochs} | Train Loss: {history['train_loss'][-1]:.4f}, Acc: {history['train_acc'][-1]:.4f} | Val Loss: {history['val_loss'][-1]:.4f}, Acc: {history['val_acc'][-1]:.4f}")


In [ ]:
# Step 5: Plot Training & Validation Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, epochs + 1), history["train_loss"], label="Train Loss", lw=2)
axes[0].plot(range(1, epochs + 1), history["val_loss"], label="Val Loss", lw=2, ls="--")
axes[0].set_title("Cross-Entropy Loss vs Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(range(1, epochs + 1), history["train_acc"], label="Train Accuracy", lw=2, color="green")
axes[1].plot(range(1, epochs + 1), history["val_acc"], label="Val Accuracy", lw=2, ls="--", color="orange")
axes[1].set_title("Accuracy vs Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Step 6: Test Set Evaluation & Confusion Matrix
model.eval()
with torch.no_grad():
    test_x_tensor = torch.from_numpy(X_test_scaled).to(device)
    test_preds = torch.argmax(model(test_x_tensor), dim=-1).cpu().numpy()

acc = np.mean(test_preds == y_test)
macro_f1 = f1_score(y_test, test_preds, average="macro")
print(f"\nTest Accuracy: {acc:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}")

action_names = [a.value for a in ACTION_INDEX_MAP.values()]
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=action_names))

cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=action_names, yticklabels=action_names)
plt.title("Test Set Confusion Matrix (11 SRE Actions)")
plt.xlabel("Predicted Action")
plt.ylabel("Ground Truth Action")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Step 7: Benchmark Single-Sample Inference Latency
sample_x = torch.from_numpy(X_test_scaled[0:1]).to(device)
latencies = []
for _ in range(1000):
    t0 = time.perf_counter()
    _ = model.predict_proba(sample_x)
    latencies.append((time.perf_counter() - t0) * 1000.0)

print(f"Single-Sample Forward Pass Latency (1,000 runs):")
print(f"  p50 (Median): {np.percentile(latencies, 50):.3f} ms")
print(f"  p95:          {np.percentile(latencies, 95):.3f} ms (Target < 2.0 ms)")
print(f"  p99:          {np.percentile(latencies, 99):.3f} ms")
